In [ ]:
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
def load_metrics(root_dir):
    data = []
    root = Path(root_dir)
    
    # Iterate through model directories
    for model_dir in root.iterdir():
        if not model_dir.is_dir():
            continue
            
        # Extract model info (e.g., "fuzzy_gat_Composite")
        # Assuming format: modelName_datasetName
        parts = model_dir.name.split('_')
        model_name = "_".join(parts[:-1])
        dataset = parts[-1]
        
        # Iterate through k directories
        for k_dir in model_dir.iterdir():
            if not k_dir.is_dir():
                continue
            
            k_val = k_dir.name # e.g., "k2"
            json_path = k_dir / "metrics.json"
            
            if json_path.exists():
                with open(json_path, 'r') as f:
                    metrics = json.load(f)
                    # Add metadata to the metrics dict
                    metrics.update({
                        'model': model_name,
                        'dataset': dataset,
                        'k': k_val
                    })
                    data.append(metrics)
                    
    return pd.DataFrame(data)

# Usage
# df = load_metrics('./root_dir')

In [ ]:
def plot_k_comparison(df, metric_name='accuracy'):
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='k', y=metric_name, hue='model')
    plt.title(f'{metric_name} Comparison across different k values')
    plt.ylabel(metric_name.capitalize())
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_best_models(df, metric_name='accuracy'):
    # Find row with max metric for each (model, dataset) combination
    idx = df.groupby(['model', 'dataset'])[metric_name].idxmax()
    best_df = df.loc[idx]
    
    plt.figure(figsize=(8, 6))
    sns.barplot(data=best_df, x='model', y=metric_name, palette='viridis')
    
    # Annotate bars with which k was the best
    for i, row in enumerate(best_df.itertuples()):
        plt.text(i, row.accuracy, f"k={row.k}", ha='center', va='bottom')
        
    plt.title(f'Comparison of Best Performing Models ({metric_name})')
    plt.tight_layout()
    plt.show()